# Risk Scoring System — Phase 2 (Bank Transaction Dataset): Build Account Profiles

**Goal:** For each account, compute a behavioural baseline ("normal") from its transaction history. The risk engine in Phase 3 scores each transaction against this baseline.

**Honest constraints carried from Phase 1 audit:**
- Median 5 transactions per account → baselines are **thin**. We compute them, but flag low-history accounts honestly.
- `PreviousTransactionDate` is a generation artifact (all 2024-11-04) → we derive velocity from `TransactionDate` instead.
- Hours are confined to a 16:00–18:00 window → time-of-day carries little signal; we still build it but it will be weighted low in Phase 3.
- No fraud label → this is pure **risk scoring** (deviation from normal), not fraud classification. That is exactly the task.

**The 8 mentor features and how each profile field supports them:**
| # | Feature | Profile field(s) |
|---|---|---|
| 1 | Avg transaction (30d) | amt_mean, amt_std |
| 2 | Max transaction ever | amt_max |
| 3 | Transactions today | avg_daily_txns, std_daily_txns |
| 4 | Typical hours | typical_hours |
| 5 | Common locations | known_locations |
| 6 | Common merchants | known_merchants |
| 7 | Account age | customer_age |
| 8 | Average balance | avg_balance, std_balance |

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import json

PATH = r"D:\Nithilan\Internship\QPay Internship\risk_scoring\data\bank_transactions_data.csv"
OUT_PROFILES = r"D:\Nithilan\Internship\QPay Internship\risk_scoring\data\account_profiles.json"

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

## 1. Load & Prepare

In [ ]:
df = pd.read_csv(PATH)
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], errors='coerce')
df['hour'] = df['TransactionDate'].dt.hour
df['date'] = df['TransactionDate'].dt.date

print(f"Loaded {len(df):,} transactions across {df['AccountID'].nunique()} accounts")
print(f"Date range: {df['TransactionDate'].min()} to {df['TransactionDate'].max()}")

## 2. Build Per-Account Profiles

For each account we compute its behavioural baseline. We use **all** of that account's transactions (there is no fraud label to exclude, and this is risk scoring not classification).

We use **population (ddof=0) standard deviation** because with as few as 5 transactions, sample std (ddof=1) is unstable and can explode. We also floor the std at a small positive value to avoid divide-by-zero in z-scores for accounts with near-constant amounts.

In [ ]:
profiles = {}
AMT_STD_FLOOR = 1.0   # avoid div-by-zero; honest minimum spread

for acct, g in df.groupby('AccountID'):
    daily = g.groupby('date').size()

    amt_std = float(g['TransactionAmount'].std(ddof=0))
    if np.isnan(amt_std) or amt_std < AMT_STD_FLOOR:
        amt_std = AMT_STD_FLOOR

    bal_std = float(g['AccountBalance'].std(ddof=0))
    if np.isnan(bal_std):
        bal_std = 0.0

    profiles[acct] = {
        'amt_mean'        : round(float(g['TransactionAmount'].mean()), 2),
        'amt_std'         : round(amt_std, 2),
        'amt_max'         : round(float(g['TransactionAmount'].max()), 2),
        'amt_median'      : round(float(g['TransactionAmount'].median()), 2),
        'avg_balance'     : round(float(g['AccountBalance'].mean()), 2),
        'std_balance'     : round(bal_std, 2),
        'min_balance'     : round(float(g['AccountBalance'].min()), 2),
        'typical_hours'   : sorted(g['hour'].dropna().unique().tolist()),
        'known_locations' : g['Location'].unique().tolist(),
        'known_merchants' : g['MerchantID'].unique().tolist(),
        'avg_daily_txns'  : round(float(daily.mean()), 2),
        'max_daily_txns'  : int(daily.max()),
        'customer_age'    : int(g['CustomerAge'].iloc[0]),
        'txn_count'       : int(len(g)),
        'low_history'     : bool(len(g) < 5),   # honest confidence flag
    }

print(f"Built {len(profiles)} account profiles")

ex_id = list(profiles.keys())[0]
ex = dict(profiles[ex_id])
ex['known_locations'] = f"{ex['known_locations']}"
ex['known_merchants'] = f"[{len(ex['known_merchants'])} merchants]"
print(f"\nExample profile (account {ex_id}):")
for k,v in ex.items():
    print(f"  {k:16}: {v}")

## 3. Sanity Checks

In [ ]:
amt_means = [p['amt_mean'] for p in profiles.values()]
locs      = [len(p['known_locations']) for p in profiles.values()]
merchs    = [len(p['known_merchants']) for p in profiles.values()]
counts    = [p['txn_count'] for p in profiles.values()]
low_hist  = [p for p in profiles.values() if p['low_history']]

print("Across all profiles:")
print(f"  amt_mean      — min {min(amt_means):.2f}, median {np.median(amt_means):.2f}, max {max(amt_means):.2f}")
print(f"  known_locs    — min {min(locs)}, median {np.median(locs):.0f}, max {max(locs)}")
print(f"  known_merch   — min {min(merchs)}, median {np.median(merchs):.0f}, max {max(merchs)}")
print(f"  txn_count     — min {min(counts)}, median {np.median(counts):.0f}, max {max(counts)}")
print(f"\nLow-history accounts (<5 txns), flagged low-confidence: {len(low_hist)} / {len(profiles)}")

## 4. Honest Note on Thin Baselines

Accounts flagged `low_history` (fewer than 5 transactions) have unreliable baselines — a "normal" computed from 1-4 transactions is statistically weak. In Phase 3, the engine should treat scores for these accounts as low-confidence rather than authoritative. We carry the flag rather than hide it.

In [ ]:
import collections
hist_dist = collections.Counter(p['txn_count'] for p in profiles.values())
print("Distribution of transactions-per-account:")
for k in sorted(hist_dist):
    bar = '#' * hist_dist[k]
    print(f"  {k:2} txns: {hist_dist[k]:3}  {bar}")

## 5. Save Profiles

In [ ]:
with open(OUT_PROFILES, 'w') as f:
    json.dump(profiles, f, indent=2)

import os
print(f"Saved {len(profiles)} profiles -> {OUT_PROFILES} ({os.path.getsize(OUT_PROFILES)/1024:.0f} KB)")

## 6. Phase 2 Summary (fill from real outputs)

| Item | Result |
|---|---|
| Profiles built | ? |
| Median locations/account | ? |
| Median merchants/account | ? |
| Low-history accounts (<5 txns) | ? |
| Balance varies within accounts | yes (from Phase 1) |

**Next — Phase 3:** build the per-feature scoring functions (0–100 each), each measuring deviation of a transaction from this account profile, then aggregate into a 0–100 risk score with a transparent breakdown. Weak features (hours) get low weight; strong ones (amount, location, balance, merchant) get high weight. Weights documented, not arbitrary.